# 🔍 Day 6B — Query the Store
**Ask → Find Relevant Chunks → Get a Cited Answer**

---

**Before running this notebook:** make sure you have run Notebook 6A
and that `store.json` exists in the same folder.

This notebook has two jobs:
1. **Search** — embed a question and find the most similar chunks
2. **Answer** — send those chunks to GPT and get a cited answer

---
## ⏱️ Time: 45 minutes

## Step 1 — Install and Import

In [ ]:
%pip install openai numpy --quiet

In [1]:
import json
import numpy as np
from getpass import getpass
from openai import AzureOpenAI

print("✅ Ready")

✅ Ready


## Step 2 — Connect to Azure OpenAI

In [2]:
AZURE_OPENAI_ENDPOINT      = input("Endpoint (e.g. https://xxx.openai.azure.com/): ").strip()
AZURE_OPENAI_API_KEY       = getpass("API Key: ")
AZURE_DEPLOYMENT_NAME      = input("Chat deployment (e.g. gpt-4o): ").strip()
AZURE_EMBEDDING_DEPLOYMENT = input("Embeddings deployment (e.g. text-embedding-ada-002): ").strip()
AZURE_API_VERSION          = "2024-08-01-preview"

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = AZURE_API_VERSION,
)
print("✅ Azure OpenAI client initialised successfully!")

Endpoint (e.g. https://xxx.openai.azure.com/):  https://agentic-ai-tt.openai.azure.com/
API Key:  ········
Chat deployment (e.g. gpt-4o):  gpt-4o
Embeddings deployment (e.g. text-embedding-ada-002):  text-embedding-ada-002


✅ Azure OpenAI client initialised successfully!


## Step 3 — Load the Store from 6A

In [3]:
with open("store.json", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"✅ Loaded {len(chunks)} chunks from store.json")
print(f"   Each chunk has: {list(chunks[0].keys())}")

✅ Loaded 5 chunks from store.json
   Each chunk has: ['id', 'text', 'vector']


## Step 4 — The Search Function

**How it works:**
1. Embed the question → get a vector
2. Compare that vector to every chunk's vector using cosine similarity
3. Return the top 3 chunks (highest similarity = most relevant)

**Cosine similarity** measures the angle between two vectors.
Score close to 1.0 means very similar meaning.
Score close to 0.0 means unrelated.

In [4]:
def cosine_similarity(vec_a, vec_b):
    """Score how similar two vectors are. Returns 0.0 to 1.0."""
    a = np.array(vec_a)
    b = np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def search(question, top_k=3):
    """Embed the question, score every chunk, return the top_k."""
    # 1. Embed the question
    q_vector = client.embeddings.create(
        model = AZURE_EMBEDDING_DEPLOYMENT,
        input = question
    ).data[0].embedding

    # 2. Score every chunk
    for chunk in chunks:
        chunk["score"] = cosine_similarity(q_vector, chunk["vector"])

    # 3. Sort by score and return the best ones
    return sorted(chunks, key=lambda c: c["score"], reverse=True)[:top_k]


print("✅ search() defined")

✅ search() defined


## Step 5 — Test the Search

Before asking GPT anything, let's verify the search is working.
We should see chunks about exports / LUT for this question.

In [5]:
question = "What GST rate applies to cloud software exported to a US company with LUT filed?"
results  = search(question)

print(f"Question: {question}\n")
print("-" * 60)
for i, chunk in enumerate(results, 1):
    print(f"Rank {i}  —  score: {chunk['score']:.3f}  (chunk #{chunk['id']})")
    print(chunk["text"][:250] + "...")
    print()

Question: What GST rate applies to cloud software exported to a US company with LUT filed?

------------------------------------------------------------
Rank 1  —  score: 0.900  (chunk #2)
Zero-Rating When LUT Is Filed A registered supplier may export IT services without payment of IGST by filing a Letter of Undertaking (LUT) in Form GST RFD-11 under Rule 96A of the CGST Rules 2017. When an LUT is filed, the applicable GST rate is 0 pe...

Rank 2  —  score: 0.846  (chunk #1)
of the IGST Act 2017. Section 3: Conditions for Export of Services IT services supplied to recipients outside India qualify as export of services under Section 13 of the IGST Act 2017 when all four conditions are met: (a) the supplier is located in I...

Rank 3  —  score: 0.844  (chunk #3)
CGST Act. Section 5: SaaS and Cloud Services Software-as-a-Service (SaaS) subscriptions and cloud computing services provided to foreign entities qualify as export of services when the four conditions in Section 3 are satisfied. 

## Step 6 — The Answer Function

Now we send the question **plus the retrieved chunks** to GPT.

The system prompt does two things:
1. Tells GPT to answer **only from the context** — not from its training memory
2. Tells GPT to **cite which chunk** the answer came from

This is what makes the answer **grounded and verifiable**.

In [6]:
def ask(question, top_k=3):
    """Search for relevant chunks then ask GPT to answer from them."""

    # Search for the most relevant chunks
    relevant = search(question, top_k=top_k)

    # Build the context block
    context = "\n\n---\n\n".join(
        f"[Chunk {chunk['id']}]\n{chunk['text']}"
        for chunk in relevant
    )

    # System prompt: answer from context only, cite the chunk
    system_prompt = (
        "You are a tax expert. Answer the question using ONLY the context below. "
        "Do not use your training knowledge. "
        "At the end of your answer, write: Source: Chunk N  "
        "(where N is the chunk number you used). "
        "If the answer is not in the context, say: 'Not found in the provided sections.'\n\n"
        f"CONTEXT:\n{context}"
    )

    response = client.chat.completions.create(
        model       = AZURE_DEPLOYMENT_NAME,
        messages    = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question},
        ],
        temperature = 0.0,
        max_tokens  = 400,
    )
    return response.choices[0].message.content, relevant


print("✅ ask() defined")

✅ ask() defined


## Step 7 — Ask Questions

In [7]:
question = "What GST rate applies to cloud software exported to a US company with LUT filed?"

answer, used_chunks = ask(question)

print(f"Q: {question}")
print(f"{'─'*60}")
print(f"A: {answer}")
print(f"{'─'*60}")
print(f"Chunks used: #{', #'.join(str(c['id']) for c in used_chunks)}")
print(f"Scores:      {', '.join(f"{c['score']:.3f}" for c in used_chunks)}")

Q: What GST rate applies to cloud software exported to a US company with LUT filed?
────────────────────────────────────────────────────────────
A: When an LUT is filed, the applicable GST rate for cloud software exported to a US company is 0 percent (zero-rated). 

Source: Chunk 2
────────────────────────────────────────────────────────────
Chunks used: #2, #1, #3
Scores:      0.900, 0.846, 0.844


In [8]:
# ← Change this and run to try your own questions
my_question = "Does reverse charge apply to legal services from an advocate?"

answer, used_chunks = ask(my_question)

print(f"Q: {my_question}")
print(f"{'─'*60}")
print(f"A: {answer}")

Q: Does reverse charge apply to legal services from an advocate?
────────────────────────────────────────────────────────────
A: Yes, reverse charge applies to legal services provided by an individual advocate to any business entity under Section 9(3) of the CGST Act 2017. The recipient (the registered business) is liable to pay GST at 18 percent, and the advocate does not collect or deposit this tax. 

Source: Chunk 3, Chunk 4.


## Step 8 — Run 3 Questions in a Row

In [9]:
questions = [
    "What is the GST rate for inter-state B2B supply of software services?",
    "What conditions must be met for a service to qualify as an export?",
    "What must be reported in GSTR-1 for export of services?",
]

for q in questions:
    answer, chunks_used = ask(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"   [chunks #{', #'.join(str(c['id']) for c in chunks_used)}]")
    print()

Q: What is the GST rate for inter-state B2B supply of software services?
A: The GST rate for inter-state B2B supply of software services is 18 percent IGST. 

Source: Chunk 0
   [chunks #0, #2, #3]

Q: What conditions must be met for a service to qualify as an export?
A: For a service to qualify as an export under Section 13 of the IGST Act 2017, the following four conditions must be met:  
(a) The supplier is located in India.  
(b) The recipient is located outside India.  
(c) The place of supply is outside India.  
(d) Payment is received in convertible foreign exchange.  

Source: Chunk 1
   [chunks #1, #3, #4]

Q: What must be reported in GSTR-1 for export of services?
A: Exports of services must be reported in Table 6A of Form GSTR-1. Additionally, a Foreign Inward Remittance Certificate (FIRC) or bank advice must be maintained as documentary evidence that payment was received in convertible foreign exchange. 

Source: Chunk 4
   [chunks #4, #1, #3]



---
## What you built across 6A and 6B

```
6A:   PDF  →  chunks  →  embed  →  store.json

6B:   question  →  embed  →  cosine similarity
                              ↓
                         top 3 chunks
                              ↓
                         GPT (reads chunks, not memory)
                              ↓
                         cited answer
```

The entire system is:
- `search()` — 8 lines
- `ask()` — 15 lines

---
**Day 7** takes the `search()` function and plugs it in as **Node 2**
of a 4-step compliance pipeline:
classify invoice → **retrieve GST rate** → calculate tax → route result.

# Faithfulness Report - Using Another LLM as a Judge

In [22]:
# ── Faithfulness scorer (LLM-as-judge) ───────────────────────────────────────
import re

FAITHFULNESS_PROMPT = """Evaluate whether each factual claim in the ANSWER is supported by the CONTEXT.

CONTEXT:
{context}

ANSWER:
{answer}

Count total claims and how many are directly supported.
Return ONLY valid JSON (no markdown):
{{"total_claims": N, "supported_claims": M, "faithfulness_score": M/N}}"""

def score_faithfulness(answer, chunks):
    context = "\n\n".join(c["text"] for c in chunks)
    prompt  = FAITHFULNESS_PROMPT.format(context=context, answer=answer)
    try:
        raw = client.chat.completions.create(
            model       = AZURE_DEPLOYMENT_NAME,
            messages    = [{"role": "user", "content": prompt}],
            temperature = 0.0,
            max_tokens  = 150,
        ).choices[0].message.content.strip()
        return json.loads(re.sub(r"^```(?:json)?\s*|\s*```$", "", raw, flags=re.MULTILINE))
    except Exception:
        return {"faithfulness_score": 0.5, "parse_error": True}

print("✅ score_faithfulness() defined")

✅ score_faithfulness() defined


In [23]:
questions = [
    "What is the GST rate for inter-state B2B supply of software services?",
    "What conditions must be met for a service to qualify as an export?",
    "What must be reported in GSTR-1 for export of services?",
]

for q in questions:
    answer, chunks_used = ask(q)
    faithfulness = score_faithfulness(answer, chunks_used)
    
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"   [chunks #{', #'.join(str(c['id']) for c in chunks_used)}]")
    print(f"faithulness: {faithfulness}")
    print()

Q: What is the GST rate for inter-state B2B supply of software services?
A: The GST rate for inter-state B2B supply of software services is 18 percent IGST. 

Source: Chunk 0
   [chunks #0, #2, #3]
faithulness: {'total_claims': 1, 'supported_claims': 1, 'faithfulness_score': 1.0}

Q: What conditions must be met for a service to qualify as an export?
A: For a service to qualify as an export under Section 13 of the IGST Act 2017, the following four conditions must be met:  
(a) The supplier is located in India.  
(b) The recipient is located outside India.  
(c) The place of supply is outside India.  
(d) Payment is received in convertible foreign exchange.  

Source: Chunk 1
   [chunks #1, #3, #4]
faithulness: {'total_claims': 4, 'supported_claims': 4, 'faithfulness_score': 1.0}

Q: What must be reported in GSTR-1 for export of services?
A: Exports of services must be reported in Table 6A of Form GSTR-1. Additionally, a Foreign Inward Remittance Certificate (FIRC) or bank advice must be

In [26]:
questions = [
    "What is the GST rate for inter-state B2B supply of software services?",
    "What conditions must be met for a service to qualify as an export?",
    "What must be reported in GSTR-1 for export of services?",
    "What is the per capita income of USA?"
]

for q in questions:
    answer, chunks_used = ask(q)
    faith = score_faithfulness(answer, chunks_used)

    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"   Faithfulness: {faith['faithfulness_score']:.2f}  "
          f"({faith.get('supported_claims','?')}/{faith.get('total_claims','?')} claims supported)")
    print()

Q: What is the GST rate for inter-state B2B supply of software services?
A: The GST rate for inter-state B2B supply of software services is 18 percent IGST. 

Source: Chunk 0
   Faithfulness: 1.00  (1/1 claims supported)

Q: What conditions must be met for a service to qualify as an export?
A: For a service to qualify as an export under Section 13 of the IGST Act 2017, the following four conditions must be met:  
(a) The supplier is located in India.  
(b) The recipient is located outside India.  
(c) The place of supply is outside India.  
(d) Payment is received in convertible foreign exchange.  

Source: Chunk 1
   Faithfulness: 1.00  (4/4 claims supported)

Q: What must be reported in GSTR-1 for export of services?
A: Exports of services must be reported in Table 6A of Form GSTR-1. Additionally, a Foreign Inward Remittance Certificate (FIRC) or bank advice must be maintained as documentary evidence that payment was received in convertible foreign exchange. 

Source: Chunk 4
   Fait